# Build Simple Query-Generation Context

This notebook creates a simplified JSONL file for LLM-based user-query generation.

Instead of passing the full CKAN/filter-parameter catalog to the LLM, each row contains only:

- dataset/package name
- dataset title
- dataset description
- resource id/name/format
- columns with lightweight metadata
- one example row, if available

Output:

```text
training/data/generated/simple_query_context.jsonl
training/data/generated/simple_query_context.csv
```

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)

RAW_DIR = Path("../training/data/raw")
GENERATED_DIR = Path("../training/data/generated")
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

FILTER_PARAMETERS_PATH = RAW_DIR / "munich_filter_parameters.jsonl"
CATALOG_PATH = RAW_DIR / "munich_catalog_sample.jsonl"
OUTPUT_JSONL = GENERATED_DIR / "simple_query_context.jsonl"
OUTPUT_CSV = GENERATED_DIR / "simple_query_context.csv"

FILTER_PARAMETERS_PATH, CATALOG_PATH

## Load source files

In [ ]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                rows.append(json.loads(line))
    return rows


filter_entries = read_jsonl(FILTER_PARAMETERS_PATH)
catalog_entries = read_jsonl(CATALOG_PATH)

print(f"filter entries: {len(filter_entries):,}")
print(f"catalog entries: {len(catalog_entries):,}")

Create a lookup so the simplified file can use the best available dataset description.

In [ ]:
catalog_by_name = {entry.get("name"): entry for entry in catalog_entries if entry.get("name")}
catalog_by_id = {entry.get("id"): entry for entry in catalog_entries if entry.get("id")}

list(catalog_by_name)[:5]

## Simplification helpers

In [ ]:
def compact_text(value: str | None, max_chars: int = 1200) -> str:
    if not value:
        return ""
    return re.sub(r"\s+", " ", value).strip()[:max_chars]


def dataset_description(entry: dict[str, Any]) -> str:
    catalog = catalog_by_name.get(entry.get("package_name")) or catalog_by_id.get(entry.get("package_id")) or {}
    return compact_text(catalog.get("notes") or entry.get("package_title"))


def simplify_column(param: dict[str, Any]) -> dict[str, Any]:
    simplified = {
        "name": param.get("column"),
        "dtype": param.get("dtype"),
        "kind": param.get("kind"),
    }

    examples = param.get("example_values") or []
    if examples:
        simplified["examples"] = examples[:5]

    if param.get("min") is not None:
        simplified["min"] = param.get("min")
    if param.get("max") is not None:
        simplified["max"] = param.get("max")

    return simplified


def simplify_example_row(entry: dict[str, Any], max_columns: int = 20) -> dict[str, Any] | None:
    sample_rows = entry.get("sample_rows") or []
    if not sample_rows:
        return None
    row = sample_rows[0]
    if not isinstance(row, dict):
        return None
    simplified = {}
    for index, (key, value) in enumerate(row.items()):
        if index >= max_columns:
            break
        if isinstance(value, str):
            value = compact_text(value, max_chars=180)
        simplified[str(key)] = value
    return simplified


def should_keep_entry(entry: dict[str, Any]) -> bool:
    if not entry.get("ok"):
        return False
    if not entry.get("filter_parameters"):
        return False
    if not entry.get("package_name") or not entry.get("resource_id"):
        return False
    return True


def simplify_entry(entry: dict[str, Any]) -> dict[str, Any]:
    columns = [simplify_column(param) for param in entry.get("filter_parameters", []) if param.get("column")]
    return {
        "package_name": entry.get("package_name"),
        "package_title": entry.get("package_title"),
        "description": dataset_description(entry),
        "resource_id": entry.get("resource_id"),
        "resource_name": entry.get("resource_name"),
        "resource_format": entry.get("resource_format"),
        "filter_mode": entry.get("filter_mode"),
        "server_filter_supported": entry.get("server_filter_supported"),
        "columns": columns,
        "example_row": simplify_example_row(entry),
    }

## Build simplified context

In [ ]:
simple_context = [simplify_entry(entry) for entry in filter_entries if should_keep_entry(entry)]

print(f"simplified resources: {len(simple_context):,}")
simple_context[0]

Inspect a compact dataframe view for sanity checking.

In [ ]:
preview_df = pd.DataFrame([
    {
        "package_name": row["package_name"],
        "package_title": row["package_title"],
        "resource_name": row["resource_name"],
        "format": row["resource_format"],
        "columns": len(row["columns"]),
        "has_example_row": row["example_row"] is not None,
        "description": row["description"],
    }
    for row in simple_context
])

preview_df.head(30)

## Save JSONL and CSV

In [ ]:
with OUTPUT_JSONL.open("w", encoding="utf-8") as file:
    for row in simple_context:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

preview_df.to_csv(OUTPUT_CSV, index=False)

OUTPUT_JSONL, OUTPUT_JSONL.stat().st_size, OUTPUT_CSV

## Suggested LLM input shape

Each JSONL row can now be passed almost directly to the query-generation LLM:

```json
{
  "package_title": "...",
  "description": "...",
  "resource_name": "...",
  "columns": [...],
  "example_row": {...}
}
```

This is much smaller and easier to reason about than the full filter-parameter catalog. Keep `package_name` and `resource_id` in the row so later generation steps can map natural queries back to retrieval/filter targets.